# Phase 4c - Product recommendation with ALS

**Goal:** suggest the **top 10 products** each user is likely to want next.

**Method:** collaborative filtering with **ALS (Alternating Least Squares)**. ALS learns a short list
of hidden "taste" numbers for every user and every product, so that
*user numbers x product numbers ~ the rating the user gave*. Products whose numbers match a
user's taste get recommended - "people who liked what you liked also liked this".

**Data:** the **official 5-core Electronics ratings** from the dataset authors (every user and product
has 5+ ratings). Phase 3 showed our 1% sample keeps only ~5,500 ratings after 5-core - too few for ALS.
The official files come already split with **leave-last-out**: each user's **last** rating is the test
item, the one before is validation, the rest is training - like predicting their next purchase.

| Step | What |
|---|---|
| 1 | Download the official train / valid / test files (~317 MB) |
| 2 | Keep a slice of users that fits free Colab (`USER_FRACTION`) |
| 3 | Turn text IDs into numbers (`StringIndexer`) |
| 4-5 | **Explicit ALS** (predicts star ratings) -> tuned, **RMSE** vs baselines |
| 6-7 | **Implicit ALS** (predicts what users choose) -> **Precision@10, HitRate@10, NDCG@10** vs "most popular" baseline |
| 8-9 | Example recommendations with product names, and save everything |

Run **top to bottom**. With `USER_FRACTION = 0.2` expect about **35-50 minutes** on free Colab (re-using the saved explicit tuning).

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/amazon_reviews'
RAW = f'{PROJECT}/raw/5core'               # official files are kept here (downloaded once)
LOCAL = '/content/data5'
FIG_DIR = f'{PROJECT}/results/figures'
OUT_DIR = f'{PROJECT}/results/recommendation'
MODEL_DIR = f'{PROJECT}/models'
BASE_URL = 'https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/benchmark/5core/last_out'
SEED = 42

USER_FRACTION = 0.2      # share of users to use. 0.2 fits free Colab; 1.0 = all (laptop / Kaggle)
N_EVAL_USERS = 3000      # users used for the top-10 evaluation (it is the slowest part)
K = 10                   # length of each recommendation list
GET_NAMES = True         # download product names for the examples (1.3 GB, about 5-10 min, once)
RUN_EXPLICIT_TUNING = False   # True = redo the 30-min explicit tuning; False = re-use saved results

In [ ]:
import os
JAVA = '/usr/lib/jvm/java-17-openjdk-amd64'
if not os.path.exists(JAVA):
    !apt-get -qq update > /dev/null && apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
os.environ['JAVA_HOME'] = JAVA
!pip -q install pyspark==3.5.3

## 1. Download the official 5-core files (only the first time)

In [ ]:
!mkdir -p {RAW} {LOCAL} {FIG_DIR} {OUT_DIR} {MODEL_DIR}
for split in ['train', 'valid', 'test']:
    target = f'{RAW}/Electronics.{split}.csv.gz'
    if not os.path.exists(target):
        print('downloading', split, '...')
        !wget -q -O /content/tmp.csv.gz {BASE_URL}/Electronics.{split}.csv.gz && mv /content/tmp.csv.gz {target}
!cp {RAW}/Electronics.*.csv.gz {LOCAL}/
!ls -lh {LOCAL}

In [ ]:
from pyspark.sql import SparkSession, Window, functions as F

spark = (SparkSession.builder
         .appName('Phase4c-ALS')
         .master('local[*]')
         .config('spark.driver.memory', '9g')
         .config('spark.sql.shuffle.partitions', '16')
         .getOrCreate())
spark.sparkContext.setLogLevel('ERROR')
spark.sparkContext.setCheckpointDir('/content/checkpoints')   # keeps ALS from building very long plans

SCHEMA = 'user_id STRING, parent_asin STRING, rating DOUBLE, timestamp LONG'
def load(split):
    return spark.read.csv(f'{LOCAL}/Electronics.{split}.csv.gz', header=True, schema=SCHEMA).repartition(16)

full = {s: load(s) for s in ['train', 'valid', 'test']}
for s, df in full.items():
    print(f'{s:<5} ratings: {df.count():>11,}')

### Helpers: chart style

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

BLUE, ORANGE, AQUA, GREY = '#2a78d6', '#eb6834', '#1baf7a', '#9a9893'
TEXT, MUTED, GRID = '#0b0b0b', '#52514e', '#e6e5e0'

plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 200, 'font.size': 10,
    'axes.edgecolor': GRID, 'axes.labelcolor': MUTED,
    'xtick.color': MUTED, 'ytick.color': MUTED,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'axes.grid.axis': 'y', 'axes.ymargin': 0.12,
    'grid.color': GRID, 'grid.linewidth': 0.8, 'axes.axisbelow': True,
    'axes.titlesize': 12, 'axes.titleweight': 'bold', 'axes.titlecolor': TEXT, 'axes.titlelocation': 'left',
    'legend.frameon': False,
})

def save_fig(fig, name):
    fig.tight_layout()
    fig.savefig(f'{FIG_DIR}/{name}.png', bbox_inches='tight', facecolor='white')
    plt.show()

## 2. Take a slice of users (so it fits free Colab)

Users are picked by a **hash of their ID**, so the same users are chosen every run, and **all**
of a chosen user's ratings (train, valid and test) are kept.

In [ ]:
def pick_users(df):
    return df.filter(F.abs(F.xxhash64('user_id')) % 1000 < int(USER_FRACTION * 1000))

train, valid, test = (pick_users(full[s]).cache() for s in ['train', 'valid', 'test'])
train_valid = train.union(valid).cache()

n_ratings = train.count() + valid.count() + test.count()
n_users = train_valid.select('user_id').distinct().count()
n_items = train.union(valid).union(test).select('parent_asin').distinct().count()
data_stats = pd.DataFrame([{
    'user_fraction': USER_FRACTION, 'ratings': n_ratings, 'users': n_users, 'products': n_items,
    'ratings_per_user': round(n_ratings / n_users, 2),
    'density_pct': round(100 * n_ratings / (n_users * n_items), 5)}])
data_stats.to_csv(f'{OUT_DIR}/data_stats.csv', index=False)
print(f'train {train.count():,}   valid {valid.count():,}   test {test.count():,}')
data_stats

**Density** = share of all (user, product) pairs that have a rating. It is tiny - each user rated
only a handful of the many products. Filling in this almost-empty table is exactly what ALS does.

## 3. Turn text IDs into numbers

ALS needs whole-number IDs. `StringIndexer` gives every user and product a number
(fitted on all three splits - only the ID list is shared, never the ratings).

In [ ]:
from pyspark.ml.feature import StringIndexer

all_ids = train.union(valid).union(test)
user_indexer = StringIndexer(inputCol='user_id', outputCol='u').fit(all_ids)
item_indexer = StringIndexer(inputCol='parent_asin', outputCol='i').fit(all_ids)

def to_index(df):
    out = item_indexer.transform(user_indexer.transform(df))
    return out.withColumn('u', F.col('u').cast('int')).withColumn('i', F.col('i').cast('int'))

train_i, valid_i, test_i, train_valid_i = (to_index(d).cache() for d in [train, valid, test, train_valid])
print(f'users indexed: {len(user_indexer.labelsArray[0]):,}   products indexed: {len(item_indexer.labelsArray[0]):,}')

## 4. Explicit ALS: tune on the validation set (rating prediction)

| Setting | Meaning | Tried |
|---|---|---|
| `rank` | how many hidden "taste" numbers per user / product | 10, 30 |
| `regParam` | how strongly the model is stopped from memorising | 0.05, 0.1, 0.3 |

Each combination is trained on **train** and scored by **RMSE on valid** (lower = better).
This takes about 30-35 minutes, so the results are **saved** and re-used on later runs
(set `RUN_EXPLICIT_TUNING = True` to redo it, e.g. after changing `USER_FRACTION`).

In [ ]:
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

rmse_eval = RegressionEvaluator(metricName='rmse', labelCol='rating', predictionCol='prediction')

def make_als(rank, reg, iters=10):
    return ALS(userCol='u', itemCol='i', ratingCol='rating', rank=rank, regParam=reg, maxIter=iters,
               nonnegative=True, coldStartStrategy='drop', seed=SEED)

TUNING_CSV = f'{OUT_DIR}/als_tuning.csv'
if RUN_EXPLICIT_TUNING or not os.path.exists(TUNING_CSV):
    tune_rows = []
    for rank in [10, 30]:
        for reg in [0.05, 0.1, 0.3]:
            t0 = time.time()
            m = make_als(rank, reg).fit(train_i)
            rmse = rmse_eval.evaluate(m.transform(valid_i))
            tune_rows.append({'rank': rank, 'regParam': reg, 'valid_rmse': round(rmse, 4),
                              'seconds': round(time.time() - t0)})
            print(f'rank={rank:<3} regParam={reg:<5} valid RMSE={rmse:.4f}  ({time.time() - t0:.0f}s)')
    tuning = pd.DataFrame(tune_rows)
    tuning.to_csv(TUNING_CSV, index=False)
else:
    tuning = pd.read_csv(TUNING_CSV)
    print('re-using saved tuning results (set RUN_EXPLICIT_TUNING = True to redo)')

best = tuning.loc[tuning['valid_rmse'].idxmin()]
BEST_RANK, BEST_REG = int(best['rank']), float(best['regParam'])
print(f'best: rank={BEST_RANK}, regParam={BEST_REG}')
tuning

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.8))
for rank, color in [(10, BLUE), (30, ORANGE)]:
    t = tuning[tuning['rank'] == rank]
    ax.plot(t['regParam'].astype(str), t['valid_rmse'], color=color, linewidth=2, marker='o',
            markersize=6, label=f'rank {rank}')
ax.set_xlabel('regParam')
ax.set_ylabel('validation RMSE (lower is better)')
ax.set_title('Explicit ALS tuning')
ax.legend(loc='upper right', ncols=2)
save_fig(fig, '4c_als_tuning')

## 5. Explicit ALS: rating accuracy (RMSE)

The final explicit model is trained on **train + valid** with the best settings and tested on the
held-out **last rating** of each user.

**Baselines (from the synopsis):**
- **Global average** - predict the same average rating for everything
- **Product average** - predict each product's average rating

In [ ]:
t0 = time.time()
model = make_als(BEST_RANK, BEST_REG).fit(train_valid_i)
train_seconds = time.time() - t0
als_rmse = rmse_eval.evaluate(model.transform(test_i))

global_mean = train_valid_i.agg(F.avg('rating')).first()[0]
item_mean = train_valid_i.groupBy('i').agg(F.avg('rating').alias('item_avg'))
base_pred = (test_i.join(item_mean, 'i', 'left')
             .withColumn('global', F.lit(global_mean))
             .withColumn('product', F.coalesce('item_avg', F.lit(global_mean))))
global_rmse = rmse_eval.setPredictionCol('global').evaluate(base_pred)
product_rmse = rmse_eval.setPredictionCol('product').evaluate(base_pred)
rmse_eval.setPredictionCol('prediction')

rmse_table = pd.DataFrame([
    {'model': 'Baseline: global average', 'test_rmse': global_rmse},
    {'model': 'Baseline: product average', 'test_rmse': product_rmse},
    {'model': f'Explicit ALS (rank {BEST_RANK}, reg {BEST_REG})', 'test_rmse': als_rmse},
]).round(4)
rmse_table.to_csv(f'{OUT_DIR}/rmse_comparison.csv', index=False)
print(f'final explicit ALS trained in {train_seconds:.0f}s')
rmse_table

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.2))
r = rmse_table.iloc[::-1]
colors = [ORANGE if 'ALS' in m else GREY for m in r['model']]
bars = ax.barh(r['model'], r['test_rmse'], height=0.55, color=colors, edgecolor='white', linewidth=1.5)
ax.bar_label(bars, labels=[f'{v:.3f}' for v in r['test_rmse']], padding=3, color=TEXT)
ax.set_xlim(0, r['test_rmse'].max() * 1.2)
ax.grid(axis='x', visible=True)
ax.grid(axis='y', visible=False)
ax.set_xlabel('test RMSE (stars, lower is better)')
ax.set_title('Rating prediction error: explicit ALS vs baselines')
save_fig(fig, '4c_rmse')

**Why explicit ALS barely beats (or even loses to) the global average:** the data is extremely
sparse - each user has ~9 ratings and each product only ~9 ratings in this slice, and most ratings
are 5 stars. With so little information per user, the best the model can do is stay close to the
average; that is also why the strongest regularisation (0.3) won the tuning.

## 6. Top-10 recommendation quality

For `N_EVAL_USERS` random users (3,000 by default) we make a top-10 list and check whether their
**real next product** is in it. Products the user already rated are removed from the list first.

| Metric | Meaning (one test product per user) |
|---|---|
| **HitRate@10** | share of users whose next product is in their top 10 |
| **Precision@10** | hits / 10 (with one test product, this is HitRate / 10) |
| **NDCG@10** | like HitRate, but a hit at rank 1 counts more than a hit at rank 10 |
| **Coverage** | how many different products appear across all lists (personalisation) |

**Baseline:** recommend the **10 most popular** products (that the user has not rated) to everyone.

In [ ]:
EXTRA = 60          # ask for more than 10, because already-rated products are removed afterwards

def top_k(cands, seen):
    '''cands: u, i, pos (lower pos = better). Drop already-rated items, keep the best K.'''
    w = Window.partitionBy('u').orderBy('pos')
    return (cands.join(seen, ['u', 'i'], 'left_anti')
                 .withColumn('rank', F.row_number().over(w))
                 .filter(F.col('rank') <= K))

def model_recs(m, users, seen):
    cands = (m.recommendForUserSubset(users, K + EXTRA)
              .select('u', F.posexplode('recommendations').alias('pos', 'rec'))
              .select('u', F.col('rec.i').alias('i'), 'pos'))
    return top_k(cands, seen)

def popular_recs(history, users, seen):
    popular = (history.groupBy('i').count().orderBy(F.desc('count')).limit(K + EXTRA)
               .withColumn('pos', F.row_number().over(Window.orderBy(F.desc('count')))).select('i', 'pos'))
    return top_k(users.crossJoin(popular), seen)

def ranking_metrics(recs, truth, name):
    hits = truth.join(recs.withColumnRenamed('i', 'true_i'), ['u', 'true_i'], 'left')
    m = hits.agg(
        F.avg(F.col('rank').isNotNull().cast('double')).alias('hit_rate'),
        F.avg(F.when(F.col('rank').isNotNull(), 1 / F.log2(F.col('rank') + 1)).otherwise(0.0)).alias('ndcg'),
    ).first()
    coverage = recs.select('i').distinct().count()
    return {'model': name, f'hit_rate@{K}': m['hit_rate'], f'precision@{K}': m['hit_rate'] / K,
            f'ndcg@{K}': m['ndcg'], 'products_recommended': coverage}

# test-set users (final comparison)
eval_users = test_i.select('u').distinct().orderBy(F.rand(SEED)).limit(N_EVAL_USERS).cache()
truth_test = test_i.join(eval_users, 'u').select('u', F.col('i').alias('true_i')).cache()
seen_test = train_valid_i.select('u', 'i').distinct().cache()

# validation-set users (for tuning implicit ALS - the test set is never used for choices)
val_users = valid_i.select('u').distinct().orderBy(F.rand(SEED + 1)).limit(N_EVAL_USERS).cache()
truth_valid = valid_i.join(val_users, 'u').select('u', F.col('i').alias('true_i')).cache()
seen_valid = train_i.select('u', 'i').distinct().cache()
print(f'evaluation users: test {eval_users.count():,}   validation {val_users.count():,}')

## 7. Implicit ALS - the right tool for "what will they buy next?"

**Why explicit ALS fails at top-10:** it learns to predict **star ratings**. A product that has only
one or two ratings, both 5 stars, gets a very high predicted rating - so the top-10 lists fill up with
**obscure products** that almost nobody buys. (Our first run: 0 hits for 3,000 users.)

**Implicit ALS** (Hu, Koren & Volinsky 2008 - already in the synopsis references) learns **which products
a user interacts with**, not how many stars they give. Every review counts as "this user chose this
product". `alpha` sets how much a real interaction counts compared with all the products the user
did not choose. We pick `alpha` on the **validation** users, then test once on the test users.

In [ ]:
IMPLICIT_RANK, IMPLICIT_REG = 20, 0.1
ALPHAS = [1.0, 10.0, 40.0]

def make_implicit(alpha, iters=10):
    return ALS(userCol='u', itemCol='i', ratingCol='bought', implicitPrefs=True, alpha=alpha,
               rank=IMPLICIT_RANK, regParam=IMPLICIT_REG, maxIter=iters,
               nonnegative=True, coldStartStrategy='drop', seed=SEED)

train_imp = train_i.withColumn('bought', F.lit(1.0))
train_valid_imp = train_valid_i.withColumn('bought', F.lit(1.0))

alpha_rows = []
for alpha in ALPHAS:
    t0 = time.time()
    m = make_implicit(alpha).fit(train_imp)
    row = ranking_metrics(model_recs(m, val_users, seen_valid), truth_valid, f'alpha={alpha:g}')
    row['alpha'] = alpha
    row['seconds'] = round(time.time() - t0)
    alpha_rows.append(row)
    print(f"alpha={alpha:<5g} validation HitRate@{K}={row[f'hit_rate@{K}']:.4f}  "
          f"NDCG@{K}={row[f'ndcg@{K}']:.4f}  ({row['seconds']}s)")

implicit_tuning = pd.DataFrame(alpha_rows)
implicit_tuning.to_csv(f'{OUT_DIR}/implicit_als_tuning.csv', index=False)
BEST_ALPHA = float(implicit_tuning.loc[implicit_tuning[f'ndcg@{K}'].idxmax(), 'alpha'])
print('best alpha =', BEST_ALPHA)

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.plot(implicit_tuning['alpha'].map('{:g}'.format), implicit_tuning[f'hit_rate@{K}'],
        color=BLUE, linewidth=2, marker='o', markersize=6)
ax.set_xlabel('alpha')
ax.set_ylabel(f'validation HitRate@{K}')
ax.set_title('Implicit ALS tuning (higher is better)')
save_fig(fig, '4c_implicit_tuning')

In [ ]:
t0 = time.time()
implicit_model = make_implicit(BEST_ALPHA).fit(train_valid_imp)
implicit_seconds = time.time() - t0
print(f'final implicit ALS trained in {implicit_seconds:.0f}s')

pop_recs = popular_recs(train_valid_i, eval_users, seen_test).cache()
explicit_recs = model_recs(model, eval_users, seen_test).cache()
implicit_recs = model_recs(implicit_model, eval_users, seen_test).cache()

ranking = pd.DataFrame([
    ranking_metrics(pop_recs, truth_test, 'Baseline: most popular'),
    ranking_metrics(explicit_recs, truth_test, 'Explicit ALS'),
    ranking_metrics(implicit_recs, truth_test, f'Implicit ALS (alpha {BEST_ALPHA:g})'),
])
ranking.to_csv(f'{OUT_DIR}/ranking_comparison.csv', index=False)
ranking.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
names = ranking['model'].tolist()
short = [n.replace(' (', '\n(') for n in names]
colors = [GREY if n.startswith('Baseline') else (BLUE if n.startswith('Implicit') else ORANGE) for n in names]
for ax, metric in zip(axes, [f'hit_rate@{K}', f'ndcg@{K}']):
    bars = ax.bar(short, ranking[metric], width=0.55, color=colors, edgecolor='white', linewidth=2)
    ax.bar_label(bars, labels=[f'{v:.4f}' for v in ranking[metric]], padding=3, color=TEXT)
    ax.set_title(metric.replace('_', ' ').replace('@', ' @ ') + ' on test users')
    ax.tick_params(axis='x', labelsize=9)
save_fig(fig, '4c_ranking')

**How to read it:** with hundreds of thousands of products, a few percent of users finding their real
next product in a 10-item list is a meaningful result. Compare the models with each other and with the
popular baseline, not the absolute number. **products_recommended** shows personalisation: the popular
baseline shows almost the same 10 products to everyone.

## 8. Example recommendations with product names

Product names come from the official metadata file (1.3 GB). It is downloaded and reduced to a
small `product_names` table **once** (set `GET_NAMES = False` at the top to skip).

In [ ]:
names_path = f'{PROJECT}/parquet/product_names'
if GET_NAMES and not os.path.exists(names_path):
    t0 = time.time()
    !wget -q -O /content/meta_Electronics.jsonl.gz https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/meta_categories/meta_Electronics.jsonl.gz
    (spark.read.schema('parent_asin STRING, title STRING, main_category STRING, average_rating DOUBLE, rating_number LONG')
          .json('/content/meta_Electronics.jsonl.gz')
          .write.mode('overwrite').parquet('/content/product_names'))
    !cp -r /content/product_names {names_path}
    !rm -f /content/meta_Electronics.jsonl.gz
    print(f'product names saved ({time.time() - t0:.0f}s)')

names = spark.read.parquet(names_path) if os.path.exists(names_path) else None
item_ids = spark.createDataFrame(list(enumerate(item_indexer.labelsArray[0])), ['i', 'parent_asin'])

In [ ]:
pd.set_option('display.max_colwidth', 80)

def with_names(df):
    out = df.join(item_ids, 'i')
    if names is not None:
        out = out.join(names.select('parent_asin', 'title', 'main_category'), 'parent_asin', 'left')
    return out

# pick an example user who rated several products
example_u = (train_valid_i.join(eval_users, 'u').groupBy('u').count()
             .filter(F.col('count') >= 6).orderBy(F.rand(SEED)).first()['u'])

print('WHAT THE USER RATED BEFORE:')
display(with_names(train_valid_i.filter(F.col('u') == example_u).select('i', 'rating', 'timestamp'))
        .orderBy('timestamp').drop('timestamp', 'i').toPandas())
print('THEIR REAL NEXT PRODUCT (hidden test item):')
display(with_names(test_i.filter(F.col('u') == example_u).select('i', 'rating')).drop('i').toPandas())
print(f'IMPLICIT ALS TOP {K}:')
display(with_names(implicit_recs.filter(F.col('u') == example_u).select('i', 'rank'))
        .orderBy('rank').drop('i').toPandas())
print(f'EXPLICIT ALS TOP {K} (for comparison - note the obscure products):')
display(with_names(explicit_recs.filter(F.col('u') == example_u).select('i', 'rank'))
        .orderBy('rank').drop('i').toPandas())

## 8b. Why these recommendations? (explainability)

Implicit ALS gives every product a **taste vector** (`rank` numbers). Two products with similar
vectors are chosen by the same kind of users. For each recommended product we find the products
**in the user's own history** with the most similar vector (cosine similarity), so the app can say:

> *Recommended because you rated **X** (similarity 0.82)*

This is the standard way to explain matrix-factorisation recommenders (item-based explanation).
We also record each product's **popularity** (ratings in training), to show whether a
recommendation is a popular product or a niche one.

In [ ]:
dot = 'aggregate(zip_with({a}, {b}, (x, y) -> cast(x * y as double)), cast(0 as double), (acc, v) -> acc + v)'
factors = implicit_model.itemFactors.select(F.col('id').alias('i'), F.col('features').alias('f'))
factors = factors.withColumn('norm', F.sqrt(F.expr(dot.format(a='f', b='f'))))

recs_f = (implicit_recs.select('u', 'i', 'rank').join(factors, 'i')
          .select('u', 'rank', F.col('i').alias('rec_i'), F.col('f').alias('rf'), F.col('norm').alias('rn')))
hist_f = (train_valid_i.join(eval_users, 'u').select('u', 'i').distinct().join(factors, 'i')
          .select('u', F.col('i').alias('hist_i'), F.col('f').alias('hf'), F.col('norm').alias('hn')))

w_best = Window.partitionBy('u', 'rec_i').orderBy(F.desc('similarity'))
because = (recs_f.join(hist_f, 'u')
           .withColumn('similarity', F.expr(dot.format(a='rf', b='hf')) / (F.col('rn') * F.col('hn')))
           .withColumn('k', F.row_number().over(w_best)).filter(F.col('k') <= 2)
           .select('u', 'rank', 'rec_i', 'hist_i', F.round('similarity', 3).alias('similarity'), 'k'))

popularity = train_valid_i.groupBy('i').count().withColumnRenamed('count', 'popularity')
print('explanation rows:', because.count())
because.orderBy('u', 'rank', 'k').show(6)

## 9. Save the models and recommendation lists

The **implicit ALS** top-10 lists of the evaluation users (with names) are saved for the **dashboard**.

In [ ]:
model.write().overwrite().save('/content/models/als_explicit_5core')
implicit_model.write().overwrite().save('/content/models/als_implicit_5core')
!mkdir -p {MODEL_DIR} && rm -rf {MODEL_DIR}/als_5core {MODEL_DIR}/als_explicit_5core {MODEL_DIR}/als_implicit_5core
!cp -r /content/models/als_explicit_5core /content/models/als_implicit_5core {MODEL_DIR}/

user_ids = spark.createDataFrame(list(enumerate(user_indexer.labelsArray[0])), ['u', 'user_id'])
dash = (with_names(implicit_recs.select('u', 'i', 'rank')).join(user_ids, 'u')
        .select('user_id', 'rank', 'parent_asin', *(['title', 'main_category'] if names is not None else []))
        .orderBy('user_id', 'rank'))
dash.write.mode('overwrite').parquet('/content/recs_dashboard')
!rm -rf {PROJECT}/parquet/als_recommendations && cp -r /content/recs_dashboard {PROJECT}/parquet/als_recommendations
print('saved both models and', dash.count(), 'recommendation rows')

# explanations for the web app: user, rank, recommended product, "because" product, similarity
asin_of = item_ids.withColumnRenamed('i', 'idx')
expl = (because.join(user_ids, 'u')
        .join(asin_of.toDF('rec_i', 'rec_asin'), 'rec_i')
        .join(asin_of.toDF('hist_i', 'because_asin'), 'hist_i')
        .join(popularity.toDF('rec_i', 'rec_popularity'), 'rec_i', 'left'))
if names is not None:
    expl = expl.join(names.select(F.col('parent_asin').alias('because_asin'), F.col('title').alias('because_title')),
                     'because_asin', 'left')
(expl.drop('u', 'rec_i', 'hist_i').write.mode('overwrite').parquet('/content/als_explanations'))
!rm -rf {PROJECT}/parquet/als_explanations && cp -r /content/als_explanations {PROJECT}/parquet/als_explanations
print('saved', expl.count(), 'explanation rows')

In [ ]:
print('=== DATA ===');              print(data_stats.to_string(index=False))
print('\n=== EXPLICIT TUNING ===');  print(tuning.to_string(index=False))
print(f'best: rank={BEST_RANK} regParam={BEST_REG}   final training {train_seconds:.0f}s')
print('\n=== RMSE ===');            print(rmse_table.to_string(index=False))
print('\n=== IMPLICIT TUNING (validation) ===')
print(implicit_tuning[['alpha', f'hit_rate@{K}', f'ndcg@{K}', 'products_recommended', 'seconds']].round(4).to_string(index=False))
print(f'best alpha={BEST_ALPHA:g}   final training {implicit_seconds:.0f}s')
print('\n=== TOP-10 (test users) ==='); print(ranking.round(4).to_string(index=False))

## Done - send back the output of the cell above

| File (in Drive) | What |
|---|---|
| `results/recommendation/als_tuning.csv` | explicit ALS tuning |
| `results/recommendation/implicit_als_tuning.csv` | implicit ALS tuning (validation) |
| `results/recommendation/rmse_comparison.csv` | RMSE vs baselines |
| `results/recommendation/ranking_comparison.csv` | Precision / HitRate / NDCG @10: popular vs explicit vs implicit ALS |
| `results/figures/4c_*.png` | 4 charts |
| `models/als_explicit_5core/`, `models/als_implicit_5core/` | the two trained models |
| `parquet/als_recommendations/` | implicit ALS top-10 lists with names (for the dashboard) |
| `parquet/als_explanations/` | "recommended because you rated X" for every list item |
| `parquet/product_names/` | names of all Electronics products |

**Next:** Phase 5 - WSL2 setup and the scalability tests.